# 01 · Coeficiente de reflexión y carta de Smith

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/telmomm/nanovna-measurement-course/HEAD?labpath=es%2Fnotebooks%2F01_reflexion_smith.ipynb)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/telmomm/nanovna-measurement-course/blob/main/es/notebooks/01_reflexion_smith.ipynb)

Un VNA te da un coeficiente de reflexión $\Gamma$ a cada frecuencia. Lo que normalmente quieres saber es una impedancia: ¿este electrodo es capacitivo?, ¿dónde resuena esta bobina?, ¿está adaptada esta antena? Ambas magnitudes están ligadas por una fórmula sencilla, y la **carta de Smith** es el dibujo de esa fórmula. En este cuaderno construyes tú la carta y aprendes a reconocer circuitos por la forma de una traza.

## Objetivos de aprendizaje

Al terminar este cuaderno serás capaz de:

- convertir entre impedancia $Z$ y coeficiente de reflexión $\Gamma$ en ambos sentidos;
- explicar qué son las circunferencias y los arcos de una carta de Smith, y situar en ella cualquier impedancia;
- reconocer en una traza el comportamiento resistivo, inductivo y capacitivo, y la resonancia;
- identificar un circuito sencillo a partir de su traza en frecuencia.

**Necesitas:** el [cuaderno 00](00_introduccion_vna.ipynb) y la impedancia de R, L y C en régimen sinusoidal permanente. Sin hardware.

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install -q git+https://github.com/telmomm/nanovna-measurement-course

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from vnacourse import plotting
from vnacourse.puzzles import mystery_trace

plotting.use_course_style()

## 1. De la impedancia a la reflexión, y vuelta

Para una carga $Z$ en un puerto de impedancia de referencia $Z_0 = 50\ \Omega$:

$$\Gamma = \frac{Z - Z_0}{Z + Z_0} \qquad\Longleftrightarrow\qquad Z = Z_0\,\frac{1 + \Gamma}{1 - \Gamma}$$

Cada impedancia corresponde exactamente a un punto del plano complejo $\Gamma$, y al revés. Escribe las dos funciones:

In [ ]:
Z0 = 50.0


def z_to_gamma(z):
    """Coeficiente de reflexión de una impedancia z en un puerto de 50 Ω."""
    z = np.asarray(z, dtype=complex)
    return (z - Z0) / (z + Z0)


def gamma_to_z(gamma):
    """Impedancia que produce el coeficiente de reflexión gamma en un puerto de 50 Ω."""
    gamma = np.asarray(gamma, dtype=complex)
    return Z0 * (1 + gamma) / (1 - gamma)


print(z_to_gamma(75))
print(gamma_to_z(0.2))

> **Predice antes de ejecutar.** ¿Dónde caen en el plano $\Gamma$ un cortocircuito (0 Ω), una carga adaptada (50 Ω), una resistencia de 150 Ω, una reactancia inductiva pura $+j50\ \Omega$ y una reactancia capacitiva pura $-j50\ \Omega$? Haz antes un esbozo en papel. El circuito abierto ($Z \to \infty$) está en $\Gamma = +1$.

In [ ]:
landmarks = {
    "cortocircuito": 0,
    "16.7 Ω": 50 / 3,
    "50 Ω": 50,
    "150 Ω": 150,
    "+j50 Ω": 50j,
    "−j50 Ω": -50j,
    "50 + j50 Ω": 50 + 50j,
}

fig, ax = plt.subplots(figsize=(5.4, 5.4))
plotting.gamma_plane(ax, radius=1.25, title="Algunas impedancias en el plano Γ")
print(f"{'impedancia':>13}  {'Γ':>16}  {'|Γ|':>5}  {'ángulo':>7}")
for name, z in landmarks.items():
    gamma = z_to_gamma(z)
    ax.plot(gamma.real, gamma.imag, "o", color=plotting.SERIES[0])
    ax.annotate(
        name, (gamma.real, gamma.imag), xytext=(0, 9), textcoords="offset points", ha="center"
    )
    print(f"{name:>13}  {gamma:16.2f}  {abs(gamma):5.2f}  {np.degrees(np.angle(gamma)):6.0f}°")
ax.plot(1, 0, "o", color=plotting.SERIES[0])
ax.annotate("abierto", (1, 0), xytext=(0, 9), textcoords="offset points", ha="center");

Cuatro cosas en las que fijarse:

- **Las resistencias están en el eje horizontal**: por debajo de 50 Ω a la izquierda, por encima de 50 Ω a la derecha y 50 Ω en el centro.
- 16.7 Ω y 150 Ω son $Z_0/3$ y $3Z_0$: caen en −0.5 y +0.5, simétricas.
- **Las reactancias puras están en la circunferencia** $|\Gamma| = 1$: lo reflejan todo, porque una reactancia no disipa potencia.
- Las impedancias **inductivas** (reactancia positiva) están en la **mitad superior** y las **capacitivas**, en la **mitad inferior**.

Todo el semiplano derecho de impedancias (cualquier impedancia con $R \ge 0$) cabe dentro de la circunferencia unidad. Esa es la gracia de trabajar con $\Gamma$: toda carga pasiva, del cortocircuito al abierto, vive en un disco finito.

## 2. Construye la carta de Smith

La carta de Smith es el plano $\Gamma$ con una retícula dibujada encima. Las líneas de la retícula son las imágenes de las líneas más sencillas del plano de impedancias:

- todas las impedancias con la **misma resistencia** $R$ y cualquier reactancia;
- todas las impedancias con la **misma reactancia** $X$ y cualquier resistencia.

> **Predice antes de ejecutar.** Una recta vertical en el plano de impedancias ($R$ fija, $X$ de $-\infty$ a $+\infty$), ¿en qué forma se convierte en el plano $\Gamma$?

In [ ]:
fig, (ax_r, ax_x) = plt.subplots(1, 2, figsize=(10, 5.2))

reactance = np.concatenate([-np.logspace(4, -2, 400), [0], np.logspace(-2, 4, 400)])
plotting.gamma_plane(ax_r, radius=1.15, title="Resistencia constante")
for resistance in (0, 16.7, 50, 150):
    gamma = z_to_gamma(resistance + 1j * reactance)
    ax_r.plot(gamma.real, gamma.imag, label=f"R = {resistance:g} Ω")
ax_r.legend(loc="upper left", bbox_to_anchor=(0.0, -0.12), ncol=2)

resistance = np.logspace(-2, 4, 400)
plotting.gamma_plane(ax_x, radius=1.15, title="Reactancia constante")
for x in (50, 150, -50, -150):
    gamma = z_to_gamma(resistance + 1j * x)
    ax_x.plot(gamma.real, gamma.imag, label=f"X = {x:+g} Ω")
ax_x.legend(loc="upper left", bbox_to_anchor=(0.0, -0.12), ncol=2);

- Las rectas de **resistencia constante** se convierten en **circunferencias**, todas tangentes en el punto $\Gamma = +1$ (el circuito abierto). $R = 0$ es la circunferencia exterior; $R = 50\ \Omega$ pasa por el centro; cuanto mayor es $R$, menores son las circunferencias, apretadas hacia la derecha.
- Las rectas de **reactancia constante** se convierten en **arcos** que también parten de $\Gamma = +1$: hacia arriba los inductivos y hacia abajo los capacitivos.

Dibuja muchas de ambas y tendrás la carta de Smith. A partir de aquí la dibuja por ti la función auxiliar del curso: `plotting.gamma_plane(ax)`. Para leer un punto, busca qué circunferencia de resistencia y qué arco de reactancia pasan por él.

Las cartas suelen rotularse con la **impedancia normalizada** $z = Z/Z_0$, de modo que la misma carta sirve para cualquier $Z_0$: la circunferencia «1» es $R = Z_0$.

## 3. Trazas: el aspecto de un circuito a lo largo de la frecuencia

Una resistencia es un punto. En cuanto hay una bobina o un condensador, la impedancia cambia con la frecuencia y el punto se mueve, dibujando una **traza**. La forma de la traza te dice el circuito.

Barremos de 10 MHz a 1.45 GHz. La función siguiente dibuja trazas y marca con un punto dónde **empieza** cada una (la frecuencia más baja).

In [ ]:
f_hz = np.linspace(10e6, 1.45e9, 600)
omega = 2 * np.pi * f_hz


def show(traces, title):
    """Dibuja trazas {etiqueta: impedancias} en una carta de Smith.

    Un punto marca la frecuencia más baja.
    """
    fig, ax = plt.subplots(figsize=(5.4, 5.4))
    plotting.gamma_plane(ax, title=title)
    for label, z in traces.items():
        gamma = z_to_gamma(z)
        (line,) = ax.plot(gamma.real, gamma.imag, label=label)
        ax.plot(gamma[0].real, gamma[0].imag, "o", color=line.get_color())
    ax.legend(loc="upper left", bbox_to_anchor=(0.0, 0.0))
    return ax

### 3.1 Circuitos serie

Una resistencia de 25 Ω en serie con una bobina, y la misma resistencia en serie con un condensador:

$$Z_{RL} = R + j\omega L \qquad\qquad Z_{RC} = R + \frac{1}{j\omega C}$$

> **Predice antes de ejecutar.** La resistencia vale 25 Ω a todas las frecuencias. ¿Por qué línea de la retícula se moverán ambas trazas? ¿En qué mitad de la carta está cada una y dónde empieza?

In [ ]:
R = 25.0
z_series_rl = R + 1j * omega * 10e-9  # 10 nH
z_series_rc = R + 1 / (1j * omega * 5e-12)  # 5 pF

show(
    {"25 Ω + 10 nH en serie": z_series_rl, "25 Ω + 5 pF en serie": z_series_rc},
    "RL y RC serie",
);

Las dos trazas se mantienen en la **circunferencia de resistencia de 25 Ω**, porque la resistencia no cambia nunca.

- La traza **RL** empieza en la resistencia (a baja frecuencia la bobina es un cortocircuito) y sube por la mitad superior, inductiva, a medida que crece $\omega L$.
- La traza **RC** empieza cerca del circuito abierto (a baja frecuencia el condensador bloquea) y baja por la mitad inferior, capacitiva, hacia la resistencia a medida que disminuye $1/\omega C$.

Ambas se mueven en **sentido horario** al aumentar la frecuencia. Esto se cumple en toda traza de un circuito pasivo hecho de R, L y C, y es una comprobación rápida de cualquier medida.

### 3.2 Circuitos paralelo

Ahora una resistencia de 100 Ω en paralelo con el mismo tipo de elemento. En paralelo se suman las admitancias:

$$\frac{1}{Z} = \frac{1}{R} + \frac{1}{j\omega L} \qquad\qquad \frac{1}{Z} = \frac{1}{R} + j\omega C$$

> **Predice antes de ejecutar.** A baja frecuencia, ¿a qué se parece una bobina en paralelo con una resistencia? ¿Y un condensador en paralelo con una resistencia a alta frecuencia?

In [ ]:
R = 100.0
z_parallel_rl = 1 / (1 / R + 1 / (1j * omega * 20e-9))  # 20 nH
z_parallel_rc = 1 / (1 / R + 1j * omega * 3e-12)  # 3 pF

show({"100 Ω ∥ 20 nH": z_parallel_rl, "100 Ω ∥ 3 pF": z_parallel_rc}, "RL y RC paralelo");

- La **RL** paralelo empieza cerca del **cortocircuito** (la bobina cortocircuita la resistencia a baja frecuencia) y recorre la mitad inductiva hacia el punto de 100 Ω.
- La **RC** paralelo empieza en el punto de 100 Ω y se dirige por la mitad capacitiva hacia el cortocircuito (el condensador cortocircuita la resistencia a alta frecuencia).

Estas trazas no siguen una circunferencia de resistencia constante. Siguen una circunferencia tangente en el *cortocircuito* en lugar de en el abierto: una circunferencia de **conductancia** constante, $G = 1/R$. Es la imagen especular de una circunferencia de resistencia. Los elementos en serie te mueven por circunferencias de resistencia; los elementos en paralelo, por circunferencias de conductancia.

### 3.3 Resonancia

Pon una bobina y un condensador en serie con una resistencia:

$$Z = R + j\left(\omega L - \frac{1}{\omega C}\right)$$

Las dos reactancias tienen signos opuestos y se anulan a la **frecuencia de resonancia** $f_0 = \dfrac{1}{2\pi\sqrt{LC}}$.

> **Predice antes de ejecutar.** ¿Dónde está la traza en la carta a $f_0$? ¿En qué mitad está por debajo de $f_0$, y por encima?

In [ ]:
R, L, C = 25.0, 20e-9, 5e-12
z_series_rlc = R + 1j * omega * L + 1 / (1j * omega * C)
f0 = 1 / (2 * np.pi * np.sqrt(L * C))

ax = show({"25 Ω + 20 nH + 5 pF en serie": z_series_rlc}, "RLC serie")
gamma_f0 = z_to_gamma(R)
ax.plot(gamma_f0.real, gamma_f0.imag, "s", color=plotting.INK)
ax.annotate("$f_0$", (gamma_f0.real, gamma_f0.imag), xytext=(-14, 8), textcoords="offset points")

fig, ax = plt.subplots()
ax.plot(f_hz / 1e6, np.abs(z_to_gamma(z_series_rlc)))
ax.axvline(f0 / 1e6, color=plotting.INK_SECONDARY, linewidth=0.8, linestyle="--")
ax.set(xlabel="frecuencia (MHz)", ylabel="$|\\Gamma|$", ylim=(0, 1.05))
ax.set_title("RLC serie: módulo de la reflexión")

print(f"Frecuencia de resonancia: {f0 / 1e6:.0f} MHz")
print(f"|Γ| en resonancia: {abs(gamma_f0):.3f}")

La traza recorre la circunferencia de 25 Ω: capacitiva (mitad inferior) por debajo de la resonancia, inductiva (mitad superior) por encima, y **cruza el eje horizontal en $f_0$**, donde la impedancia es solo la resistencia. En la gráfica del módulo, la resonancia es el mínimo: la reflexión es menor donde se anula la reactancia.

Leer una resonancia en la carta: el cruce con el eje real te da a la vez $f_0$ y la resistencia. La gráfica del módulo solo te da $f_0$.

### 3.4 Una resistencia tras un cable

Una forma más, del cuaderno 00: una resistencia de 75 Ω al final de 20 cm de cable. El cable no cambia $|\Gamma|$; solo gira la fase.

In [ ]:
tau = 0.20 / (0.7 * 299_792_458.0)
gamma_cable = z_to_gamma(75) * np.exp(-4j * np.pi * f_hz * tau)

ax = show(
    {"75 Ω tras 20 cm de cable": gamma_to_z(gamma_cable)},
    "Un retardo dibuja una circunferencia en torno al centro",
)

z_seen = gamma_to_z(gamma_cable)
print(f"Resistencia vista en el puerto: de {z_seen.real.min():.1f} Ω a {z_seen.real.max():.1f} Ω")

Una circunferencia **centrada en el centro de la carta** significa $|\Gamma|$ constante y fase que gira: la firma de un tramo de línea delante del dispositivo. A través del cable, la resistencia de 75 Ω parece cualquier cosa entre 33 Ω y 75 Ω, con reactancia, según la frecuencia. Te encontrarás exactamente esto en datos reales en el cuaderno 03.

| Forma de la traza | Circuito |
|---|---|
| un punto en el eje horizontal | resistencia |
| arco por una circunferencia de resistencia, mitad superior | R en serie con L |
| arco por una circunferencia de resistencia, mitad inferior | R en serie con C |
| arco por una circunferencia de conductancia (tangente en el cortocircuito) | R en paralelo con L o C |
| cruza el eje horizontal | resonancia |
| circunferencia en torno al centro | un tramo de línea delante del dispositivo |

## 4. Adivina el circuito

Tres circuitos misteriosos, cada uno formado por una resistencia y uno o dos elementos reactivos. `mystery_trace("A", f_hz)` devuelve el coeficiente de reflexión del circuito A. Usa lo que has aprendido:

1. ¿Dónde empieza y dónde acaba la traza? ¿Qué es el circuito a muy baja y a muy alta frecuencia?
2. ¿Sigue una circunferencia de resistencia o de conductancia?
3. ¿Cruza el eje horizontal?

Después convierte a impedancia con `gamma_to_z` y lee los valores. Decide la topología *antes* de calcular nada.

In [ ]:
for name in ("A", "B", "C"):
    gamma = mystery_trace(name, f_hz)
    fig, ax = plt.subplots(figsize=(4.6, 4.6))
    plotting.gamma_plane(ax, title=f"Circuito misterioso {name}")
    ax.plot(gamma.real, gamma.imag, color=plotting.SERIES[0])
    ax.plot(gamma[0].real, gamma[0].imag, "o", color=plotting.SERIES[0])

In [ ]:
# Averigua aquí los circuitos. Por ejemplo, la impedancia de A a la frecuencia más alta:
z_a = gamma_to_z(mystery_trace("A", f_hz))
print(f"Z de A a {f_hz[-1] / 1e6:.0f} MHz: {z_a[-1]:.1f} Ω")

## Comprueba lo que has entendido

1. ¿Dónde está $Z = 50 - j50\ \Omega$ en la carta? Da $|\Gamma|$ y la mitad de la carta.

<details><summary>Respuesta</summary>

$\Gamma = \dfrac{-j50}{100 - j50} = 0.2 - 0.4j$, así que $|\Gamma| = 0.45$, en la mitad inferior (capacitiva), sobre la circunferencia $R = 50\ \Omega$ que pasa por el centro.
</details>

2. Una traza empieza en el cortocircuito, recorre la mitad superior de la carta y acaba en el eje horizontal, en el punto de 100 Ω. ¿Qué circuito es?

<details><summary>Respuesta</summary>

Una resistencia de 100 Ω en paralelo con una bobina. A baja frecuencia la bobina cortocircuita la resistencia (la traza empieza en el cortocircuito); a alta frecuencia la bobina es un circuito abierto y solo queda la resistencia. La mitad superior confirma un elemento inductivo, y el camino es una circunferencia de conductancia, como en todo elemento en paralelo.
</details>

3. ¿Por qué todas las trazas de circuitos pasivos quedan dentro de la circunferencia unidad?

<details><summary>Respuesta</summary>

$|\Gamma| > 1$ significaría más potencia reflejada que recibida. Un circuito pasivo tiene $R \ge 0$, y toda impedancia con $R \ge 0$ cae dentro o sobre la circunferencia $|\Gamma| = 1$.
</details>

4. En la gráfica del módulo del RLC serie, ¿cómo cambiaría el mínimo si la resistencia fuera de 50 Ω en lugar de 25 Ω?

<details><summary>Respuesta</summary>

En resonancia $Z = R = 50\ \Omega$, así que $\Gamma = 0$: el mínimo llegaría hasta cero. La traza pasaría por el centro de la carta. Es un circuito resonante adaptado.
</details>

## Ejercicios

Las soluciones resueltas están en [`exercises/solutions/01_solutions.ipynb`](../exercises/solutions/01_solutions.ipynb).

1. **Ida y vuelta.** Calcula a mano $\Gamma$ para $Z = 20 - j30\ \Omega$, compruébalo con `z_to_gamma` y vuelve a convertir. Después halla la impedancia en $\Gamma = 0.5\angle 90°$.
2. **Admitancia.** Comprueba numéricamente que el coeficiente de reflexión de la carga reflejada en admitancia, $Z' = Z_0^2 / Z$, es $-\Gamma$: el mismo punto girado 180°. Úsalo para explicar por qué las circunferencias de conductancia son imágenes especulares de las de resistencia.
3. **Resonancia paralelo.** Dibuja la traza de 100 Ω ∥ 5 nH ∥ 20 pF. ¿Dónde está en resonancia? ¿En qué mitad está por debajo de la resonancia: la opuesta al caso serie, o la misma?
4. **Los circuitos misteriosos.** Para A, B y C da la topología y estima los valores de los componentes a partir de la traza.

In [ ]:
# Tu trabajo aquí

## Resumen

- $\Gamma = (Z - Z_0)/(Z + Z_0)$ lleva toda impedancia pasiva al disco unidad: cortocircuito en −1, adaptación en 0, abierto en +1.
- La carta de Smith es ese disco con circunferencias de resistencia constante y arcos de reactancia constante.
- Mitad superior inductiva, mitad inferior capacitiva, eje horizontal resistivo, borde puramente reactivo.
- Las trazas de circuitos pasivos se mueven en sentido horario con la frecuencia. Los elementos en serie siguen circunferencias de resistencia, los elementos en paralelo siguen circunferencias de conductancia, y una resonancia cruza el eje horizontal.
- Una circunferencia en torno al centro es un retardo: una línea delante del dispositivo.

**Siguiente:** en [02 · Errores sistemáticos](02_errores_sistematicos.ipynb) descubrirás que la $\Gamma$ que muestra un instrumento no es la $\Gamma$ del dispositivo.